# DINOv2 üç komşu seçimini iyileştiriyor mu?
Her turda bir toprağın bütün çekimlerini saklıyoruz. Altı doku ölçüsü ile hazır DINOv2'nin görüntüden çıkardığı 384 sayıyı karşılaştırıyoruz. Her iki yöntem de seçtiği üç toprağın laboratuvar eğrilerinin ortalamasını kullanıyor.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.preprocessing import StandardScaler

root = Path("/kaggle/input") if Path("/kaggle/input").exists() else next(p for p in [Path("work/soil-data"), Path("../work/soil-data")] if p.exists())
data = next(root.rglob("Training_labels_updated.csv")).parent
labels = pd.read_csv(data / "Training_labels_updated.csv").set_index("sample_id")
cameras = pd.read_csv(data / "ppm_updated.csv")
photos = sorted(data.joinpath("Training-All_Photos_updated").rglob("*.jpg"))
print(f"{len(labels)} numune, {len(photos)} eğitim fotoğrafı")

24 numune, 127 eğitim fotoğrafı


## 1. Fotoğrafı bir avuç sayıyla anlatmak
Bütün fotoğrafların merkezinden yaklaşık 100 × 100 mm alan alıyoruz. Kamera tablosundaki piksel/mm değerini gerçek dosyanın çözünürlüğüne uyarlıyoruz. Bu kalibrasyon, çekim uzaklığının kamera tablosunda varsayılana yakın olduğunu kabul eder.

Bu alanı griye çevirip 400 × 400 piksele getiriyoruz. Yatay ve dikey yönde, birbirinden 0,25 mm ile 8 mm arasında uzakta olan noktaların ne kadar farklı aydınlıkta göründüğünü ölçüyoruz. Böylece her fotoğrafta aynı altı fiziksel uzaklığı karşılaştırıyoruz. Her numunenin fotoğraflarını, her ölçünün ortanca değerini alarak birleştiriyoruz.

In [2]:
def physical_patch(path, camera):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        w, h = image.size
        pixels_per_mm = camera.ppm * max(w, h) / max(camera.width, camera.height)
        side = round(100 * pixels_per_mm)
        assert 0 < side <= min(w, h)
        x, y = (w - side) // 2, (h - side) // 2
        return image.crop((x, y, x + side, y + side)).convert("RGB").resize((400, 400), Image.Resampling.LANCZOS)

def texture(patch):
    gray = np.asarray(patch.convert("L"), dtype=float)
    low, high = np.percentile(gray, [5, 95])
    gray = np.clip((gray - low) / max(high - low, 1.0), 0, 1)
    values = []
    for offset in [1, 2, 4, 8, 16, 32]:
        horizontal = np.abs(gray[:, offset:] - gray[:, :-offset]).mean()
        vertical = np.abs(gray[offset:, :] - gray[:-offset, :]).mean()
        values.append((horizontal + vertical) / 2)
    return values

camera_by_name = {c.phone.replace(" ", "_").lower(): c for c in cameras.itertuples()}
rows = []
examples = {}
for path in photos:
    phone, sample, number = path.stem.rsplit("_", 2)
    patch = physical_patch(path, camera_by_name[phone.lower()])
    rows.append([sample, *texture(patch)])
    examples.setdefault(sample, patch)
features = pd.DataFrame(rows, columns=["sample_id", "0.25", "0.5", "1", "2", "4", "8"])
features = features.groupby("sample_id").median()
assert set(features.index) == set(labels.index)
features = features.loc[labels.index]

## 2. Gerçek cevabı saklayıp tahmin yapmak
Her turda yalnızca geride kalan 23 numuneden ölçekleme değerlerini öğreniyoruz. Altı doku ölçüsüne göre uzaklık hesaplayıp en yakın üç komşuyu seçiyoruz. Komşuların ölçümlerinin ortalaması tahminimiz oluyor. Karşılaştırma için aynı 23 numunenin tamamının ortalamasını da hesaplıyoruz.

In [3]:
sizes = labels.columns.astype(float).to_numpy()
widths = np.diff(np.log10(sizes))

def emd(observed, predicted):
    return np.abs(observed[:-1] - predicted[:-1]) @ widths

scores, neighbors, predictions, averages = [], {}, {}, {}
for sample in labels.index:
    train = features.drop(index=sample)
    scaler = StandardScaler().fit(train)
    distance = np.linalg.norm(scaler.transform(train) - scaler.transform(features.loc[[sample]]), axis=1)
    nearest = train.index[np.argsort(distance, kind="stable")[:3]].tolist()
    predicted = labels.loc[nearest].mean().to_numpy()
    baseline = labels.loc[train.index].mean().to_numpy()
    observed = labels.loc[sample].to_numpy()
    assert sample not in nearest and len(set(nearest)) == 3
    assert np.all(np.diff(predicted) >= -1e-10) and np.isclose(predicted[-1], 100)
    scores.append((sample, emd(observed, baseline), emd(observed, predicted)))
    neighbors[sample], predictions[sample], averages[sample] = nearest, predicted, baseline
scores = pd.DataFrame(scores, columns=["Numune", "Ortalama", "Üç komşu"]).set_index("Numune")
print(scores.mean().round(2).to_string())
print(f"Üç komşunun daha az hata yaptığı numune sayısı: {(scores['Üç komşu'] < scores['Ortalama']).sum()}/{len(scores)}")

Ortalama    89.89
Üç komşu    44.74
Üç komşunun daha az hata yaptığı numune sayısı: 18/24


## Her fotoğraftan 384 ölçü

DINOv2-S/14 önceden eğitilmiş bir görüntü modeli. Ağırlıklarını sabit tutuyoruz. Modelin tüm görüntüyü özetleyen vektörünü (CLS) alıyoruz. Aynı 100 mm genişliğindeki renkli kesiti 224 × 224 piksele indiriyoruz. Doku yöntemi aynı kesiti 400 × 400 pikselde gri tonlarda işler. Fark bu iki görüntü işleme yolunun toplam etkisini ölçer.

Yerelde çalışma klasöründeki ağırlık dosyası kullanılır. Kaggle'da aynı dosyayı `dinov2_small.safetensors` adıyla girdi olarak ekleyebilir veya internet açıkken modelin indirilmesini sağlayabilirsiniz. [Model ağırlıkları](https://huggingface.co/timm/vit_small_patch14_dinov2.lvd142m)

In [4]:
import hashlib
import timm
import torch
from safetensors.torch import load_file

torch.set_num_threads(min(4, torch.get_num_threads()))
name = "vit_small_patch14_dinov2.lvd142m"
paths = [Path("work/dinov2_small.safetensors"), Path("../work/dinov2_small.safetensors")]
if Path("/kaggle/input").exists():
    paths.extend(Path("/kaggle/input").rglob("dinov2_small.safetensors"))
weights = next((p for p in paths if p.is_file()), None)
if weights is None:
    from huggingface_hub import hf_hub_download
    weights = Path(hf_hub_download("timm/" + name, "model.safetensors"))
with weights.open("rb") as f:
    assert hashlib.file_digest(f, "sha256").hexdigest() == "04d27f3400d059fc0cfd7d17dd1909a75bf3ea8fb3eeb48b97cb99e57ee20081"
model = timm.create_model(name, pretrained=False, dynamic_img_size=True, num_classes=0)
model.load_state_dict(load_file(str(weights)), strict=True)
model.eval()
mean = torch.tensor(model.pretrained_cfg["mean"]).view(3, 1, 1)
std = torch.tensor(model.pretrained_cfg["std"]).view(3, 1, 1)
ids, vectors = [], []
with torch.inference_mode():
    for start in range(0, len(photos), 8):
        batch = []
        for path in photos[start:start + 8]:
            phone, sample, _ = path.stem.rsplit("_", 2)
            patch = physical_patch(path, camera_by_name[phone.lower()])
            rgb = np.array(patch.resize((224, 224), Image.Resampling.BICUBIC))
            tensor = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255
            batch.append((tensor - mean) / std)
            ids.append(sample)
        vectors.extend(model(torch.stack(batch)).numpy())
embeddings = pd.DataFrame(vectors, index=ids).groupby(level=0).median().loc[labels.index]
assert embeddings.shape == (24, 384) and np.isfinite(embeddings).all().all()
print(len(ids), "fotoğraf,", len(embeddings), "toprak,", embeddings.shape[1], "özellik")

127 fotoğraf, 24 toprak, 384 özellik


## Aynı 24 toprakta iki yöntemi karşılaştır

Her turda uzaklık ölçeğini yalnızca diğer 23 topraktan öğreniyoruz. Saklanan toprağın laboratuvar eğrisine tahmini yaptıktan sonra bakıyoruz. Pozitif “Kazanç” DINOv2'nin hatayı kaç puan azalttığını gösterir. H038 için H371 ve H617'nin görüntü sırasını da kaydediyoruz.

In [5]:
records = []
for sample in labels.index:
    train = embeddings.drop(index=sample)
    scaler = StandardScaler().fit(train)
    distance = np.linalg.norm(scaler.transform(train) - scaler.transform(embeddings.loc[[sample]]), axis=1)
    order = train.index[np.argsort(distance, kind="stable")].tolist()
    chosen = order[:3]
    assert sample not in chosen and len(set(chosen)) == 3
    predicted = labels.loc[chosen].mean().to_numpy()
    assert np.all(np.diff(predicted) >= -1e-10) and np.isclose(predicted[-1], 100)
    observed = labels.loc[sample].to_numpy()
    err = emd(observed, predicted)
    old = scores.loc[sample, "Üç komşu"]
    lab = sorted(train.index, key=lambda other: emd(observed, labels.loc[other].to_numpy()))[:3]
    records.append([sample, old, err, old - err, len(set(chosen) & set(lab)), ", ".join(chosen)])
    if sample == "H038":
        print("H038 için görüntü sırası:", {s: order.index(s) + 1 for s in ["H371", "H617"]})
report = pd.DataFrame(records, columns=["Numune", "Doku", "DINOv2", "Kazanç", "Ortak", "Yeni komşular"]).set_index("Numune")
assert len(report) == 24 and abs(report.Doku.mean() - 44.744) < 0.001
print(report.round(2).to_string())
print("Ortalama EMD:", report[["Doku", "DINOv2"]].mean().round(3).to_dict())
print("Daha iyi / eşit / daha kötü:", int((report.Kazanç > 1e-8).sum()), int((report.Kazanç.abs() <= 1e-8).sum()), int((report.Kazanç < -1e-8).sum()))
print("Laboratuvarın ilk üçüyle ortak seçim:", report.Ortak.sum(), "/ 72")

H038 için görüntü sırası: {'H371': 15, 'H617': 2}
          Doku  DINOv2  Kazanç  Ortak     Yeni komşular
Numune                                                 
F827     63.98   93.73  -29.76      1  H038, H030, H549
G190     24.85   24.85    0.00      1  F827, H038, H030
H030     44.43   21.96   22.46      1  H666, H617, H549
H037     60.27  109.72  -49.45      0  H617, H405, H126
H038     93.28  148.81  -55.53      0  H549, H617, H668
H126     14.28  101.37  -87.09      0  H037, H617, H038
H181     20.89   17.66    3.23      2  H183, H366, H367
H183     28.41   13.28   15.14      2  H366, H367, H181
H366      8.11   10.02   -1.91      2  H367, H183, H371
H367      7.67   12.59   -4.92      1  H366, H183, H181
H368     14.50   19.16   -4.66      1  H371, H366, H372
H371     14.03    8.02    6.00      2  H372, H366, H368
H372      7.27    5.46    1.80      2  H371, H368, H366
H374    128.87   19.78  109.10      2  H366, H367, H183
H405    133.53  133.53    0.00      0  H037, H617, H03